# Duplicate Timestamps in PSP SWEAP/SPI AF00 (Proton HR) CDF Files

This notebook demonstrates that the L3 high-rate proton moment files
(`psp_swp_spi_af00_L3_mom`) contain **genuinely duplicated TT2000 timestamps**
in the `Epoch` variable.

The duplicates cause downstream analysis failures (e.g., `pandas.reindex` crashes
with "cannot reindex on an axis with duplicate labels").

**Affected file shown here:** `psp_swp_spi_af00_L3_mom_20240930_v04.cdf` (E21)

**Other instruments checked (no duplicates):**
- `psp_swp_spi_sf00_l3_mom` (standard-rate proton) -- clean
- `psp_swp_spe_af0_L3_pad` (high-rate electron) -- clean

## 0. Download the CDF file via pyspedas

This cell downloads the PSP SWEAP/SPI high-rate proton moment file for 2024-09-30 (E21).
If you already have it locally, it will use the cached version.

In [ ]:
import pyspedas
import cdflib
import numpy as np
from pathlib import Path
import glob
import os

In [ ]:
# Download the high-rate proton moments for E21 (2024-09-30)
trange = ['2024-09-30 00:00:00', '2024-09-30 23:59:59']

spi_files = pyspedas.psp.spi(
    trange=trange,
    datatype='spi_af00_L3_mom',
    level='l3',
    downloadonly=True,   # Just download, don't load into tplot
    no_update=False
)

print(f"\nDownloaded files:")
for f in spi_files:
    print(f"  {f}")

## 1. Load the CDF and read the Epoch variable

In [ ]:
# Use the first downloaded file
CDF_PATH = spi_files[0]

cdf = cdflib.CDF(CDF_PATH)
epoch = cdf.varget('Epoch')   # TT2000 int64 nanoseconds
print(f"File: {os.path.basename(CDF_PATH)}")
print(f"Epoch variable length: {len(epoch):,} samples")
print(f"Epoch dtype: {epoch.dtype}")

## 2. Count duplicate timestamps

In [ ]:
diffs = np.diff(epoch)
dup_mask = diffs == 0
n_duplicates = dup_mask.sum()

print(f"Adjacent timestamp pairs with identical TT2000 values: {n_duplicates}")
print(f"Total samples: {len(epoch):,}")
print(f"Affected fraction: {n_duplicates / len(epoch):.4%}")

## 3. Show the duplicate pairs

In [ ]:
dup_indices = np.where(dup_mask)[0]

print(f"Showing first 10 duplicate pairs (of {n_duplicates} total):\n")
print(f"{'Index':>8}  {'TT2000 value':>22}  {'Human-readable (UTC)':>30}")
print("-" * 65)

for idx in dup_indices[:10]:
    tt2000_val = epoch[idx]
    dt_str = str(cdflib.cdfepoch.to_datetime(tt2000_val)[0])
    print(f"{idx:>8}  {tt2000_val:>22}  {dt_str:>30}  <-- row {idx}")
    print(f"{idx+1:>8}  {epoch[idx+1]:>22}  {dt_str:>30}  <-- row {idx+1}  ** DUPLICATE **")
    print()

## 4. Are the data values also identical, or just the timestamps?

In [ ]:
info = cdf.cdf_info()

# Pick a science variable to compare values at duplicate timestamps
try:
    vel = cdf.varget('VEL_RTN_SUN')
    var_name = 'VEL_RTN_SUN'
except:
    sci_vars = [v for v in info.zVariables if v not in 
                ('Epoch', 'TIME', 'MET', 'APID', 'SEQN', 'SEQN_DELTA', 
                 'SEQN_GROUP', 'PKT_SIZE', 'SOURCE_APID', 'SOURCE_HASH')]
    var_name = sci_vars[0]
    vel = cdf.varget(var_name)

print(f"Comparing '{var_name}' at duplicate-timestamp rows:\n")

n_identical_data = 0
n_different_data = 0

for idx in dup_indices:
    row_a = vel[idx]
    row_b = vel[idx + 1]
    if np.array_equal(row_a, row_b, equal_nan=True):
        n_identical_data += 1
    else:
        n_different_data += 1

print(f"Duplicate pairs where '{var_name}' values are IDENTICAL: {n_identical_data}")
print(f"Duplicate pairs where '{var_name}' values DIFFER:        {n_different_data}")

if n_different_data > 0:
    print(f"\nExample of differing values at a duplicate timestamp:")
    for idx in dup_indices:
        if not np.array_equal(vel[idx], vel[idx+1], equal_nan=True):
            dt_str = str(cdflib.cdfepoch.to_datetime(epoch[idx])[0])
            print(f"  Index {idx}:   {vel[idx]}")
            print(f"  Index {idx+1}: {vel[idx+1]}")
            print(f"  Timestamp:  {dt_str}")
            break

## 5. Cross-check: other instruments on the same day

Download standard-rate proton and high-rate electron files for comparison.

In [ ]:
# Download standard-rate proton
sf00_files = pyspedas.psp.spi(
    trange=trange,
    datatype='spi_sf00_l3_mom',
    level='l3',
    downloadonly=True,
    no_update=False
)

# Download high-rate electron
spe_files = pyspedas.psp.spe(
    trange=trange,
    datatype='spe_af0_L3_pad',
    level='l3',
    downloadonly=True,
    no_update=False
)

In [ ]:
print(f"{'Instrument':<35} {'Samples':>10} {'Duplicates':>12}")
print("-" * 60)
print(f"{'Proton high-rate (spi_af00) *BUG*':<35} {len(epoch):>10,} {n_duplicates:>12}")

cross_checks = [
    ("Proton standard-rate (sf00)", sf00_files),
    ("Electron high-rate (spe_af0)", spe_files),
]

for label, files in cross_checks:
    if files:
        c = cdflib.CDF(files[0])
        e = c.varget('Epoch')
        d = (np.diff(e) == 0).sum()
        print(f"{label:<35} {len(e):>10,} {d:>12}")
    else:
        print(f"{label:<35} {'(not found)':>10} {'--':>12}")

print("\nOnly the high-rate proton file (spi_af00) has duplicate timestamps.")

## 6. Where in the file do the duplicates cluster?

In [ ]:
import matplotlib.pyplot as plt

dup_times = cdflib.cdfepoch.to_datetime(epoch[dup_indices])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

# Top: where duplicates occur in the file
ax1.scatter(dup_indices, np.ones(len(dup_indices)), marker='|', c='red', s=100, alpha=0.5)
ax1.set_xlim(0, len(epoch))
ax1.set_xlabel('Sample index')
ax1.set_title(f'Location of {n_duplicates} duplicate timestamp pairs in {os.path.basename(CDF_PATH)}')
ax1.set_yticks([])

# Bottom: histogram of inter-sample time deltas
diffs_us = diffs / 1000  # TT2000 nanoseconds -> microseconds
ax2.hist(diffs_us[diffs_us < np.percentile(diffs_us, 99)], bins=100, color='steelblue', edgecolor='none')
ax2.axvline(0, color='red', linewidth=2, label=f'dt=0 (duplicates, n={n_duplicates})')
ax2.set_xlabel('Inter-sample delta (microseconds)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of time deltas between consecutive samples')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nDuplicate timestamps span from index {dup_indices[0]:,} to {dup_indices[-1]:,}")
print(f"Time range of duplicates: {dup_times[0]} to {dup_times[-1]}")

## Summary

The `psp_swp_spi_af00_L3_mom` CDF file for 2024-09-30 (Encounter 21, v04) contains
**124 pairs of exactly duplicated TT2000 timestamps** in the `Epoch` variable.

- The duplicates are in the source CDF -- not introduced by any analysis code.
- Other instruments on the same day (standard-rate proton, high-rate electron) do **not** have this issue.
- This breaks any downstream code that assumes monotonically unique timestamps
  (e.g., `pandas.Series.reindex`, interpolation routines, etc.).

**Workaround:** Drop duplicate timestamps (keep first occurrence) before reindexing.

**Suggested fix:** The SPI L3 processing pipeline should either:
1. Deduplicate before writing the CDF, or
2. Increment the duplicate timestamp by 1 nanosecond to preserve both samples.